<table style="width: 100%; border: none; background: linear-gradient(to bottom, #f8f9fa, #e9ecef); padding: 20px; border-radius: 10px; box-shadow: 0 2px 5px rgba(0,0,0,0.1);">
    <tr>
        <td style="width: 22%; border: none; vertical-align: middle; text-align: center;">
            <img src="https://www.inf.ucv.cl/wp-content/uploads/2020/05/logo_escuela.jpg" width="170" style="max-width: 100%;">
        </td>
        <td style="width: 56%; border: none; vertical-align: middle; text-align: center; padding: 0 20px;">
            <h1 style="font-size: 28px; color: #1a3b5c; margin-bottom: 16px; border-bottom: 2px solid #1a3b5c; padding-bottom: 8px;">Tarea 2</h1>
            <p style="font-size: 16px; margin: 8px 0;"><strong>Pontificia Universidad Católica de Valparaíso</strong><br>
            <strong>Escuela de Ingeniería Informática</strong></p>
            <p style="font-size: 14px; color: #333; margin: 12px 0;"><strong>ICD 3152 Estadística Avanzada</strong><br>
            <strong>Semestre: 2-2025</strong><br>
            <strong>Profesor: Carlos Valle</strong><br>
            <strong>Fecha: 15 de octubre de 2025</strong></p>
        </td>
        <td style="width: 42%; border: none; vertical-align: middle; text-align: center;">
            <img src="https://navegador.pucv.cl/imagen/Logo-color(1).png" width="280" style="max-width: 100%;">
        </td>
    </tr>
</table>

# Tarea 2: Ridge, Bootstrap y Validación de Modelos

**Objetivo:** Aplicar regresión Ridge, métodos de remuestreo (Bootstrap/Jackknife) y técnicas de validación cruzada para construir y evaluar modelos predictivos robustos.

**Contexto:** Predecir el promedio final de estudiantes universitarios usando información académica previa, socioeconómica y de dedicación.

**Formalidades**  
* Entregar Jupyter Notebook (.ipynb) con análisis, interpretaciones y conclusiones en celdas de texto (markdown).
* Fecha de Entrega: **Martes 4 de noviembre, 23:59 horas**
* La Nota se calcula: $\left(\frac{\mbox{suma de puntos obtenidos}*6}{76}\right)+1$

<hr style="height:2px;border:none"/>


# Parte A: Regresión Ridge y Multicolinealidad


## 1.1 (4 pts) Análisis exploratorio y diagnóstico de multicolinealidad

**Contexto teórico:** La multicolinealidad ocurre cuando variables predictoras están altamente correlacionadas, causando:
- Alta varianza en los coeficientes OLS: $\text{Var}(\hat{\boldsymbol{\beta}}_{OLS}) = \sigma^2(\mathbf{X}^T\mathbf{X})^{-1}$.
- Matriz $\mathbf{X}^T\mathbf{X}$ mal condicionada (valores propios pequeños).
- Inestabilidad: pequeños cambios en datos → grandes cambios en $\hat{\boldsymbol{\beta}}$

**Implementar:**
1. Cargar los datos `datos_rendimiento_universitario.csv`
2. Realizar análisis descriptivo básico (dimensiones, tipos, valores faltantes).
3. Calcular la matriz de correlación de todas las variables predictoras.
4. Visualizar con heatmap las correlaciones lineales.
5. Calcular el **Variance Inflation Factor (VIF)** para cada predictor:
   $$VIF_j = \frac{1}{1 - R_j^2},$$
   donde $R_j^2$ es el $R^2$ de la regresión de $X_j$ sobre todas las demás variables.
   
6. Identificar variables con $VIF > 10$ (multicolinealidad severa) o $VIF > 5$ (moderada).
7. Calcular el **número de condición** $\kappa(\mathbf{X}^T\mathbf{X})$ usando `np.linalg.cond()`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuración
np.random.seed(3152)
plt.style.use('default')

# Su código aquí - cargar datos y análisis exploratorio


## 1.2 (3 pts) Interpretación del diagnóstico de multicolinealidad

**Responda basándose en sus resultados:**
1. ¿Qué pares de variables presentan las correlaciones más altas? ¿Tiene sentido desde el contexto del problema?
2. ¿Cuáles variables tienen VIF > 10? ¿Qué implica esto para OLS?
3. ¿Qué indica el número de condición $\kappa$ sobre la estabilidad de la matriz $\mathbf{X}^T\mathbf{X}$?
4. Dado este diagnóstico, ¿por qué es apropiado usar Ridge en este problema?

## 2.1 (5 pts) Regresión Ridge con selección de λ

**Contexto teórico:** Ridge minimiza:
$$\min_{\boldsymbol{\beta}} \left\{ \|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|^2 + \lambda\|\boldsymbol{\beta}\|^2 \right\}$$

Solución: $\hat{\boldsymbol{\beta}}_{Ridge} = (\mathbf{X}^T\mathbf{X} + \lambda\mathbf{I})^{-1}\mathbf{X}^T\mathbf{y}$

**Efecto de λ:**
- $\lambda = 0$: recupera OLS (sin penalización)
- $\lambda$ grande: coeficientes se encogen hacia 0 (alto sesgo, baja varianza)
- Mejora número de condición: $\kappa(\mathbf{X}^T\mathbf{X} + \lambda\mathbf{I}) \ll \kappa(\mathbf{X}^T\mathbf{X})$

**Implementar:**
1. Dividir datos en Train (80%) y Test (20%) usando `train_test_split`
2. **Estandarizar** variables (crucial para Ridge): centrar y escalar en Train, aplicar misma transformación en Test
3. Ajustar modelo OLS en Train y evaluar MSE en Test (baseline)
4. Crear grid de λ: `lambdas = np.logspace(-2, 4, 100)` (desde 0.01 hasta 10,000)
5. Para cada λ:
   - Ajustar Ridge en Train
   - Calcular MSE en Train y en Test
   - Guardar $\|\hat{\boldsymbol{\beta}}\|^2$ (norma L2 de coeficientes)
6. Graficar:
   - MSE Train vs MSE Test en función de log(λ)
   - Coeficientes vs log(λ) (Ridge trace plot)
7. Identificar $\lambda^*$ óptimo (mínimo MSE en Test)
8. Calcular mejora en número de condición: $\kappa(\mathbf{X}^T\mathbf{X} + \lambda^*\mathbf{I})$ vs $\kappa(\mathbf{X}^T\mathbf{X})$

In [ ]:
# Su código aquí - Regresión Ridge


## 2.2 (4 pts) Interpretación de Ridge

**Responda basándose en sus resultados:**
1. ¿Cuál es el valor de $\lambda^*$ óptimo? ¿Cómo cambia el MSE de Test comparado con OLS?
2. En el Ridge trace plot, ¿qué variables se encogen más rápido cuando λ aumenta? ¿Por qué?
3. ¿En cuánto mejora el número de condición con Ridge? ¿Qué implica esto para la estabilidad?
4. Explique el trade-off entre sesgo y varianza observado: ¿por qué Ridge supera a OLS en Test a pesar de introducir sesgo?

## 3.1 (4 pts) Análisis de estabilidad: Ridge vs OLS

**Contexto teórico:** Un modelo es **estable** si pequeños cambios en los datos no cambian drásticamente los coeficientes estimados. Ridge debería ser más estable que OLS cuando hay multicolinealidad.

**Experimento de perturbación:**
- Agregamos ruido gaussiano pequeño a los datos
- Medimos cuánto cambian los coeficientes

**Implementar:**
1. Usar los datos de Train estandarizados

2. Ajustar OLS y Ridge (con $\lambda^*$) y guardar coeficientes originales

3. Realizar 50 perturbaciones:
   - Agregar ruido: $X_{\text{pert}} = X + \mathcal{N}(0, 0.05 \cdot \sigma_X)$ donde $\sigma_X$ es la std de cada variable
   - Reajustar OLS y Ridge en datos perturbados
   - Calcular cambio relativo: $\frac{\|\hat{\boldsymbol{\beta}}_{\text{nuevo}} - \hat{\boldsymbol{\beta}}_{\text{original}}\|_2}{\|\hat{\boldsymbol{\beta}}_{\text{original}}\|_2}$

4. Comparar:
   - Distribución de cambios relativos (boxplot OLS vs Ridge)
   - Cambio promedio y desviación estándar
   - Para cada coeficiente individual, comparar su variabilidad entre OLS y Ridge

In [ ]:
# Su código aquí - Análisis de estabilidad


## 3.2 (3 pts) Interpretación de estabilidad

**Responda:**
1. ¿Qué método muestra menor cambio relativo promedio en los coeficientes? ¿En qué porcentaje?
2. ¿Qué coeficientes específicos son más estables con Ridge vs OLS?
3. Relacione estos resultados con el número de condición calculado en 1.1: ¿la mejora en estabilidad coincide con la mejora en $\kappa$?
4. ¿Por qué es importante la estabilidad para la confianza en el modelo en producción?

# Parte B: Métodos de Remuestreo

## 4.1 (5 pts) Bootstrap para intervalos de confianza de coeficientes Ridge

**Contexto teórico:** El bootstrap estima la distribución de un estimador mediante remuestreo:

**Algoritmo Bootstrap No Paramétrico:**
1. Para $b = 1, \ldots, B$:
   - Generar muestra bootstrap: muestrear **con reemplazo** $n$ observaciones de los datos originales
   - Ajustar modelo en muestra bootstrap
   - Guardar coeficientes $\hat{\boldsymbol{\beta}}^*_b$
   
2. Usar $\{\hat{\boldsymbol{\beta}}^*_1, \ldots, \hat{\boldsymbol{\beta}}^*_B\}$ para:
   - Estimar error estándar: $\hat{se}(\hat{\beta}_j) = \sqrt{\frac{1}{B-1}\sum_{b=1}^B (\hat{\beta}_{j,b}^* - \bar{\beta}_j^*)^2}$
   - Construir IC percentil: $[\hat{\beta}_{j,(\alpha/2)}^*, \hat{\beta}_{j,(1-\alpha/2)}^*]$

**Implementar:**
1. Usar los datos de Train estandarizados
2. Ajustar modelo Ridge con $\lambda^*$ (del ejercicio 2.1)
3. Realizar $B = 1000$ réplicas bootstrap:
   - En cada réplica: `indices = np.random.choice(n, n, replace=True)`
   - Ajustar Ridge en `X_train[indices]`, `y_train[indices]`
   - Guardar coeficientes
4. Para cada coeficiente $\hat{\beta}_j$:
   - Calcular error estándar bootstrap
   - Construir IC percentil 95%
   - Visualizar histograma de la distribución bootstrap
5. Crear tabla con: Variable, Coef. original, SE bootstrap, IC 95%
6. Identificar qué coeficientes tienen IC que no incluyen 0

In [ ]:
# Su código aquí - Bootstrap para coeficientes Ridge


## 4.2 (3 pts) Interpretación del Bootstrap de coeficientes

**Responda:**
1. ¿Qué coeficientes tienen IC que **no** incluyen el 0? ¿Qué implica esto sobre su importancia?
2. ¿Las distribuciones bootstrap son aproximadamente normales o muestran asimetría?
3. ¿Por qué el bootstrap es útil para Ridge, donde no hay fórmulas cerradas estándar para los errores estándar?
4. Compare los errores estándar bootstrap entre variables con VIF alto vs bajo (del ejercicio 1.1). ¿Qué observa?

## 5.1 (5 pts) Bootstrap BCa para R²

**Contexto teórico:** El **intervalo percentil simple** asume simetría. Para distribuciones sesgadas o estadísticos no-lineales como $R^2$, el método **BCa (Bias-Corrected and Accelerated)** corrige por:

1. **Sesgo ($\hat{z}_0$)**: mide qué tan sesgada está la distribución bootstrap
   $$\hat{z}_0 = \Phi^{-1}\left(\frac{\#\{\hat{\theta}^*_b < \hat{\theta}\}}{B}\right)$$

2. **Aceleración ($\hat{a}$)**: mide curvatura, usa jackknife:
   $$\hat{a} = \frac{\sum_{i=1}^n (\bar{\theta}_{(\cdot)} - \hat{\theta}_{(i)})^3}{6[\sum_{i=1}^n (\bar{\theta}_{(\cdot)} - \hat{\theta}_{(i)})^2]^{3/2}}$$

3. **Cuantiles ajustados**:
   $$\alpha_1 = \Phi\left(\hat{z}_0 + \frac{\hat{z}_0 + z_{\alpha/2}}{1 - \hat{a}(\hat{z}_0 + z_{\alpha/2})}\right)$$
   
**Implementar:**
1. Calcular $R^2$ observado en Test para Ridge con $\lambda^*$

2. **Bootstrap** ($B = 2000$):
   - En cada réplica: remuestrear Train, ajustar Ridge, evaluar en Test
   - Guardar $\{R^2_1^*, \ldots, R^2_B^*\}$

3. **IC Percentil simple**: percentiles 2.5% y 97.5%

4. **IC BCa**:
   - Calcular $\hat{z}_0$: proporción de réplicas bootstrap < $R^2$ observado
   - Calcular $\hat{a}$ usando jackknife:
     - Para cada $i$: ajustar Ridge sin observación $i$ en Train, calcular $R^2_{(i)}$ en Test
     - Aplicar fórmula de aceleración
   - Calcular $\alpha_1, \alpha_2$ ajustados con $z_{0.025} = -1.96$, $z_{0.975} = 1.96$
   - IC BCa: percentiles $\alpha_1$ y $\alpha_2$ de las réplicas bootstrap

5. Visualizar:
   - Histograma de distribución bootstrap de $R^2$
   - Marcar $R^2$ observado y ambos intervalos

In [ ]:
# Su código aquí - Bootstrap BCa para R²
from scipy.stats import norm



## 5.2 (3 pts) Interpretación de BCa

**Responda:**
1. ¿Los IC percentil y BCa difieren significativamente? ¿En qué dirección?
2. ¿Qué indica el signo de $\hat{z}_0$? ¿La distribución bootstrap de $R^2$ es simétrica o sesgada?
3. Basándose en el IC BCa, ¿qué tan bueno es el poder predictivo del modelo Ridge? ¿El intervalo es estrecho o amplio?
4. ¿Por qué es importante usar BCa para $R^2$ en lugar del percentil simple?

## 6.1 (4 pts) Jackknife vs Bootstrap para MSE

**Contexto teórico:** El **Jackknife** elimina una observación a la vez:
- Para $i = 1, \ldots, n$: calcular $\hat{\theta}_{(i)}$ usando datos sin observación $i$
- Error estándar: $\hat{se}_J = \sqrt{\frac{n-1}{n}\sum_{i=1}^n (\hat{\theta}_{(i)} - \bar{\theta}_{(\cdot)})^2}$

**Comparación:**
- Jackknife: $n$ réplicas (determinístico)
- Bootstrap: $B$ réplicas (aleatorio)

**Implementar:**
1. Estadístico de interés: MSE de Ridge (con $\lambda^*$) en Test

2. **Jackknife:**
   - Para cada $i = 1, \ldots, n$ en Train:
     - Ajustar Ridge sin observación $i$
     - Predecir en Test y calcular $MSE_{(i)}$
   - Calcular $\bar{MSE}_{(\cdot)} = \frac{1}{n}\sum_{i=1}^n MSE_{(i)}$
   - Calcular $\hat{se}_J = \sqrt{\frac{n-1}{n}\sum_{i=1}^n (MSE_{(i)} - \bar{MSE}_{(\cdot)})^2}$

3. **Bootstrap:**
   - Realizar $B = 1000$ réplicas
   - En cada réplica: remuestrear Train, ajustar Ridge, calcular MSE en Test
   - Calcular $\hat{se}_B$ de las réplicas bootstrap

4. Comparar:
   - Errores estándar: $\hat{se}_J$ vs $\hat{se}_B$
   - Tiempo computacional (usar `%%time` o `time.time()`)
   - Visualizar: boxplot bootstrap vs puntos jackknife

In [ ]:
# Su código aquí - Jackknife vs Bootstrap
import time



## 6.2 (3 pts) Interpretación Jackknife vs Bootstrap - CONTINUACIÓN

**Responda:**
1. ¿Los errores estándar de Jackknife y Bootstrap son similares o difieren significativamente?
2. ¿Cuál método fue más rápido? ¿Por cuánto factor?
3. ¿Cuándo preferirías Jackknife sobre Bootstrap para este tipo de análisis?
4. ¿Qué método consideras más confiable para estimar la variabilidad del MSE y por qué?

# Parte C: Validación Cruzada y Comparación de Modelos

## 7.1 (5 pts) Validación Cruzada k-fold para Ridge vs OLS

**Contexto teórico:** La validación cruzada k-fold particiona los datos en $k$ subconjuntos (folds):

1. Para cada fold $i = 1, \ldots, k$:
   - Entrenar en $k-1$ folds
   - Validar en fold $i$
   - Calcular error $\hat{R}_i$

2. Estimador CV: $\hat{R}_{CV}(k) = \frac{1}{k}\sum_{i=1}^k \hat{R}_i$

**Propiedades:**
- Cada observación se usa exactamente **una vez** para validación
- Estimación menos sesgada del error de generalización que Train/Test simple
- Trade-off: $k = 5$ o $k = 10$ es típico (balance sesgo-varianza-costo computacional)

**Implementar:**
1. Usar **todos los datos** (no hacer split Train/Test)

2. Realizar 10-fold CV para:
   - **OLS** (sin hiperparámetros)
   - **Ridge** con $\lambda^*$ del ejercicio 2.1
   
3. Para cada método:
   - Usar `cross_val_score` con `scoring='neg_mean_squared_error'` y `cv=10`
   - Guardar los 10 MSE individuales de cada fold
   - Calcular MSE promedio: $\overline{MSE}_{CV}$
   - Calcular error estándar: $se(\overline{MSE}_{CV}) = \frac{sd(MSE_i)}{\sqrt{k}}$

4. Visualizar:
   - Boxplot de los 10 MSE de cada método lado a lado
   - Barplot con MSE medio ± SE para cada método

5. **Test pareado:**
   - Calcular diferencias: $d_i = MSE_{OLS,i} - MSE_{Ridge,i}$ para cada fold
   - Realizar t-test pareado: $t = \frac{\bar{d}}{s_d/\sqrt{k}}$ con $df = k-1$
   - Calcular p-valor
   - ¿Es la diferencia estadísticamente significativa al 5%?

In [ ]:
# Su código aquí - Validación cruzada k-fold
from sklearn.model_selection import cross_val_score, KFold
from scipy.stats import ttest_rel



## 7.2 (3 pts) Interpretación de validación cruzada

**Responda:**
1. ¿Ridge tiene menor MSE promedio que OLS? ¿Por cuánto?
2. ¿Qué método muestra mayor variabilidad entre folds? ¿Qué implica sobre su robustez?
3. ¿El test pareado indica que la diferencia es estadísticamente significativa? Interprete el p-valor.
4. ¿Por qué usar todos los datos (no split Train/Test) da una estimación más confiable del error de generalización?

## 8.1 (7 pts) Nested Cross-Validation para comparar múltiples algoritmos

**Contexto teórico:** Cuando comparamos métodos donde cada uno requiere ajustar hiperparámetros, usar CV simple produce **sesgo optimista**.

**Problema:** Si usamos el mismo validation set para:
1. Seleccionar hiperparámetros óptimos
2. Comparar los métodos

→ El "ganador" pudo haber tenido suerte en ese validation set

**Solución: Nested CV** (dos niveles independientes)

```
Para cada fold externo k = 1,...,K:
    ├─ Separar: Dev_k (80%) y Test_k (20%)
    │
    ├─ Loop interno sobre Dev_k:
    │   └─ CV para encontrar hiperparámetros óptimos
    │       (λ para Ridge, arquitectura para MLP, etc.)
    │
    ├─ Entrenar modelo final con hiperparámetros óptimos en todo Dev_k
    │
    └─ Evaluar HONESTAMENTE en Test_k → error e_k

Error final honesto: MSE_nested = (1/K) Σ e_k
```

**Comparar 4 algoritmos:**
- **Ridge Regression** (hiperparámetro: λ)
- **MLP (Multi-Layer Perceptron)** (hiperparámetros: hidden layers, learning rate)
- **Random Forest** (hiperparámetros: n_estimators, max_depth)
- **CatBoost** (hiperparámetros: iterations, learning_rate, depth)

**Implementar:**

1. **Loop externo:** 5-fold CV sobre todos los datos

2. Para cada fold externo $k = 1, \ldots, 5$:
   
   **A. Ridge:**
   - Loop interno: `GridSearchCV` con 5-fold CV sobre Dev_k
   - Grid: `lambdas = [0.01, 0.1, 1, 10, 100, 1000]`
   - Encontrar $\lambda^*_k$ óptimo
   - Entrenar Ridge($\lambda^*_k$) en todo Dev_k
   - Evaluar en Test_k → $MSE_{Ridge,k}$
   
   **B. MLP:**
   - Grid: `{'hidden_layer_sizes': [(50,), (100,), (50,50)], 'learning_rate_init': [0.001, 0.01]}`
   - `MLPRegressor(max_iter=500, random_state=3152)`
   - GridSearchCV con 5-fold
   - Entrenar mejor configuración en Dev_k
   - Evaluar en Test_k → $MSE_{MLP,k}$
   
   **C. Random Forest:**
   - Grid: `{'n_estimators': [50, 100, 200], 'max_depth': [5, 10, None]}`
   - `RandomForestRegressor(random_state=3152)`
   - GridSearchCV con 5-fold
   - Evaluar en Test_k → $MSE_{RF,k}$
   
   **D. CatBoost:** (OPCIONAL - si tienen instalado)
   - Grid: `{'iterations': [100, 200], 'learning_rate': [0.03, 0.1], 'depth': [4, 6]}`
   - Si no tienen CatBoost, pueden usar **GradientBoostingRegressor** de sklearn
   - Evaluar en Test_k → $MSE_{CB,k}$

3. Calcular MSE promedio honesto para cada método:
   $$\overline{MSE}_{\text{Ridge}}^{nested} = \frac{1}{5}\sum_{k=1}^5 MSE_{Ridge,k}$$

4. Crear tabla comparativa:
   - Método | MSE Nested CV | SE | Ranking

5. Visualizar:
   - Boxplot de los 5 MSE de cada método
   - Comparar con CV simple del ejercicio 7.1

In [ ]:
# Su código aquí - Nested Cross-Validation
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
# from catboost import CatBoostRegressor  # Opcional si está instalado

# ADVERTENCIA: Este código puede tomar 5-10 minutos en ejecutar
# debido al doble loop de CV



## 8.2 (4 pts) Interpretación de Nested CV

**Responda:**
1. ¿Qué algoritmo tiene el menor MSE en Nested CV? ¿Es Ridge el ganador o algún método de ML?
2. Compare MSE de Ridge en CV simple (ejercicio 7.1) vs Nested CV. ¿Hay diferencia? ¿Cuánto **sesgo optimista** había?
3. ¿Los hiperparámetros óptimos encontrados en cada fold externo son consistentes o varían mucho?
4. Después de este análisis riguroso, ¿qué método recomendarías para producción y por qué?

## 9.1 (4 pts) Análisis de estabilidad predictiva con Bootstrap

**Contexto:** Queremos evaluar qué tan estables son las predicciones de cada modelo cuando hay variabilidad en los datos de entrenamiento.

**Método:** Para un conjunto de observaciones de test fijas, realizar bootstrap sobre train y ver cuánto varían las predicciones.

**Implementar:**
1. Crear split Train (80%) / Test (20%) fijo

2. Seleccionar 10 observaciones representativas de Test (variedad de valores de y)

3. Para Ridge, MLP y Random Forest:
   - Realizar B = 100 réplicas bootstrap:
     - Remuestrear Train con reemplazo
     - Ajustar modelo en muestra bootstrap (con hiperparámetros del ejercicio 8.1)
     - Predecir en las 10 observaciones de Test
   - Para cada observación de test:
     - Calcular desviación estándar de las 100 predicciones
     - Calcular rango (max - min)

4. Comparar:
   - Promedio de desviaciones estándar por método
   - Promedio de rangos por método
   - Visualizar: boxplot de desviaciones estándar de predicciones para cada método

5. Para una observación específica, graficar:
   - Histograma de las 100 predicciones de cada método

In [ ]:
# Su código aquí - Estabilidad predictiva


## 9.2 (3 pts) Interpretación de estabilidad predictiva

**Responda:**
1. ¿Qué método produce predicciones más estables (menor desviación estándar promedio)?
2. ¿Hay diferencia entre la estabilidad de predicciones para observaciones con y bajo vs y alto?
3. ¿Por qué es importante la estabilidad predictiva además de la precisión (MSE bajo)?
4. ¿Cómo se relaciona esta estabilidad predictiva con la estabilidad de coeficientes analizada en el ejercicio 3.1?

## 10 (4 pts) Conclusiones finales

**Sintetice sus hallazgos respondiendo:**

1. **Sobre multicolinealidad y Ridge:**
   - ¿Cómo el diagnóstico de multicolinealidad justificó el uso de Ridge?
   - ¿En qué porcentaje mejoró la estabilidad Ridge vs OLS?
   - ¿Hubo trade-off significativo en términos de MSE?

2. **Sobre métodos de remuestreo:**
   - ¿Qué valor agregaron Bootstrap y Jackknife al análisis?
   - ¿Cuándo fue crucial usar BCa en lugar de percentil simple?
   - ¿Cuál método (Bootstrap/Jackknife) prefiere para este tipo de problemas?

3. **Sobre validación y comparación de modelos:**
   - ¿Por qué Nested CV es crucial para comparaciones honestas?
   - ¿Cuánto sesgo optimista detectó en CV simple?
   - ¿Qué modelo final recomendaría para producción?

4. **Recomendación final:**
   - Si tuviera que implementar UN modelo en producción para predecir promedio_final, ¿cuál elegiría considerando:
     * Precisión (MSE)
     * Estabilidad
     * Interpretabilidad
     * Costo computacional
   - Justifique su decisión con evidencia cuantitativa de la tarea.